# Jvalyx model training (Google Colab)

Trains the two models the backend loads and saves them to your Google Drive:

| File | Model | Loaded by |
|---|---|---|
| `catboost_model.cbm` | CatBoost 5-class incident classifier (12 features) | `backend/pipeline/inference.py` |
| `isolation_forest.joblib` | Isolation Forest anomaly scorer (5 features) | `backend/pipeline/inference.py` |

**Before running**
1. Upload `data/training/jvalyx_labeled_firms_india_v3.csv` to Google Drive at `MyDrive/jvalyx/`.
2. *Runtime → Change runtime type → T4 GPU* (CatBoost uses it; Isolation Forest is CPU-only and fast anyway).
3. *Runtime → Run all*.

**After running** — the models are in `MyDrive/jvalyx/artifacts/` and zipped as `MyDrive/jvalyx/jvalyx_artifacts.zip`.
Download the zip, then on your laptop from the repo root:

```
python backend/models/artifacts/install_artifacts.py --from "C:\Users\<you>\Downloads\jvalyx_artifacts.zip"
```

Library versions are pinned to the backend's (`catboost 1.2.10`, `scikit-learn 1.9.0`) so the files load without version warnings.

## 1. Install pinned libraries

In [ ]:
!pip install -q catboost==1.2.10 scikit-learn==1.9.0 joblib==1.4.2
# If Colab asks you to restart the runtime after this, restart and continue from section 2.

In [ ]:
import shutil, subprocess, catboost, sklearn, joblib, numpy as np, pandas as pd
print('catboost', catboost.__version__, '| sklearn', sklearn.__version__, '| joblib', joblib.__version__,
      '| numpy', np.__version__, '| pandas', pd.__version__)
assert catboost.__version__ == '1.2.10', 'catboost version must match the backend'
assert sklearn.__version__ == '1.9.0', 'scikit-learn version must match the backend (restart runtime if you just installed)'
USE_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
if not USE_GPU:
    print('No GPU: CatBoost will train on CPU (much slower). Runtime -> Change runtime type -> T4 GPU.')
print('GPU available:', USE_GPU)

## 2. Mount Drive and configure

In [ ]:
import os, shutil, time
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/jvalyx'      # where you uploaded the CSV
CSV_NAME = 'jvalyx_labeled_firms_india_v3.csv'
OUT_DIR = f'{DRIVE_DIR}/artifacts'               # trained models are written here

CAP_RATIO = 3             # cap every class at CAP_RATIO x the smallest class (flares, ~73.5k)
GRID_DEG = 0.05           # ~5 km cells; each cell goes wholly to train OR validation (no location leakage)
IFOREST_ROWS = 2_000_000  # random detections used to fit the Isolation Forest
RUN_LEAKAGE_CHECK = True  # also train a model WITHOUT facility_type/lulc_class, for comparison only
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)
src = f'{DRIVE_DIR}/{CSV_NAME}'
assert os.path.exists(src), f'Upload {CSV_NAME} to MyDrive/jvalyx/ first'
LOCAL_CSV = f'/content/{CSV_NAME}'
if not os.path.exists(LOCAL_CSV) or os.path.getsize(LOCAL_CSV) != os.path.getsize(src):
    t = time.time()
    shutil.copy(src, LOCAL_CSV)  # reading 2.4 GB straight off Drive is much slower
    print(f'Copied CSV to local disk in {time.time() - t:.0f}s')
print(f'{os.path.getsize(LOCAL_CSV) / 1e9:.2f} GB')

## 3. Load the dataset

In [ ]:
import gc

# Exact order the backend builds its feature row in (backend/pipeline/inference.py FEATURE_NAMES).
# recurrence_days_90d (distinct earlier days in the last 90 with a detection in the same ~1.1 km
# cell) is what separates routine site heat (C5) from accidental/unusual site fires (C1) in v3.
FEATURE_NAMES = [
    'bright_ti4', 'bright_ti5', 'temp_ratio', 'frp', 'scan', 'track', 'daynight',
    'is_in_industrial_polygon', 'distance_to_industrial_m', 'facility_type', 'lulc_class', 'lulc_entropy_500m',
    'recurrence_days_90d',
]
CATEGORICAL = ['facility_type', 'lulc_class']   # indices 9 and 10
IFOREST_FEATURES = ['bright_ti4', 'bright_ti5', 'temp_ratio', 'frp', 'distance_to_industrial_m']
CLASS_NAMES = {1: 'Unusual industrial fire', 2: 'Wildfire', 3: 'Mining fire', 4: 'Stubble burning', 5: 'Routine industrial heat/flare'}

usecols = FEATURE_NAMES + ['class_id', 'label_source', 'latitude', 'longitude']
t = time.time()
df = pd.read_csv(LOCAL_CSV, usecols=usecols, engine='pyarrow')
df = df[usecols]  # usecols keeps the file's column order; enforce ours
for c in ['facility_type', 'label_source']:
    df[c] = df[c].astype('category')
df['is_in_industrial_polygon'] = df['is_in_industrial_polygon'].astype(bool).astype('int8')
print(f'Loaded {len(df):,} rows in {time.time() - t:.0f}s, {df.memory_usage(deep=True).sum() / 1e9:.1f} GB in RAM')

In [ ]:
# Isolation Forest is unsupervised: sample from ALL detections (labeled or not) before filtering.
if_df = df[IFOREST_FEATURES].dropna()
if_df = if_df.sample(n=min(IFOREST_ROWS, len(if_df)), random_state=SEED).astype('float64')
print('Isolation Forest sample:', if_df.shape)

lab = df.dropna(subset=['class_id'] + FEATURE_NAMES)
del df; gc.collect()
lab = lab.copy()
lab['class_id'] = lab['class_id'].astype('int8')
lab['daynight'] = lab['daynight'].astype('int8')
# The backend sends land cover as float-formatted strings ("40.0"); train on exactly that.
lulc = lab['lulc_class'].astype('float32').astype('category')
lab['lulc_class'] = lulc.cat.rename_categories([f'{c:.1f}' for c in lulc.cat.categories])
print(f'Labeled rows: {len(lab):,}')
print(lab['class_id'].value_counts().sort_index().rename(CLASS_NAMES))

## 4. Balance classes (downsample only — no duplicated or synthetic rows)

In [ ]:
counts = lab['class_id'].value_counts().sort_index()
cap = int(counts.min() * CAP_RATIO)
rng = np.random.default_rng(SEED)
picked = [rng.choice(idx, size=min(len(idx), cap), replace=False)
          for _, idx in sorted(lab.groupby('class_id', observed=True).indices.items())]
bal = lab.iloc[np.concatenate(picked)].reset_index(drop=True)
del lab; gc.collect()

print(f'Cap per class: {cap:,}')
print(bal['class_id'].value_counts().sort_index().rename(CLASS_NAMES))
print(bal['label_source'].value_counts())

## 5. Spatial train/validation split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# The same mine/flare/farm burns thousands of times. Keeping each ~5 km cell on one side stops the
# validation score from being inflated by near-duplicate detections of the same site.
groups = np.floor(bal['latitude'] / GRID_DEG).astype(np.int64) * 100_000 + np.floor(bal['longitude'] / GRID_DEG).astype(np.int64)
train_idx, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(bal, bal['class_id'], groups))
train, val = bal.iloc[train_idx], bal.iloc[val_idx]
print(f'train {len(train):,} rows / {groups.iloc[train_idx].nunique():,} cells | val {len(val):,} rows / {groups.iloc[val_idx].nunique():,} cells')
print(pd.DataFrame({'train': train['class_id'].value_counts(), 'val': val['class_id'].value_counts()}).sort_index().rename(CLASS_NAMES))
assert set(val['class_id']) == {1, 2, 3, 4, 5}, 'a class is missing from validation; change SEED or GRID_DEG'

## 6. Train CatBoost

In [ ]:
from catboost import CatBoostClassifier, Pool

def to_pool(part, features=FEATURE_NAMES):
    X = part[features].copy()
    cats = [c for c in CATEGORICAL if c in features]
    for c in cats:
        X[c] = X[c].astype(str)
    return Pool(X, label=part['class_id'].to_numpy(), cat_features=cats)

train_pool, val_pool = to_pool(train), to_pool(val)

params = dict(
    loss_function='MultiClass', eval_metric='MultiClass',
    iterations=3000, learning_rate=0.08, depth=8,
    auto_class_weights='SqrtBalanced', early_stopping_rounds=200, use_best_model=True,
    random_seed=SEED, verbose=200,
)
if USE_GPU:
    params.update(task_type='GPU', devices='0')

t = time.time()
model = CatBoostClassifier(**params)
model.fit(train_pool, eval_set=val_pool)
print(f'Trained in {(time.time() - t) / 60:.1f} min, best iteration {model.get_best_iteration()}')

## 7. Validation metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

y_val = val['class_id'].to_numpy()
pred = model.predict(val_pool).astype(int).ravel()
labels = [1, 2, 3, 4, 5]
report_text = classification_report(y_val, pred, labels=labels, target_names=[CLASS_NAMES[c] for c in labels], digits=3)
print(report_text)
cm = pd.DataFrame(confusion_matrix(y_val, pred, labels=labels),
                  index=[f'true {CLASS_NAMES[c]}' for c in labels], columns=[f'pred {c}' for c in labels])
display(cm)
macro_f1 = f1_score(y_val, pred, average='macro')
print(f'Macro F1: {macro_f1:.3f}')

imp = pd.Series(model.get_feature_importance(train_pool), index=FEATURE_NAMES).sort_values(ascending=False)
display(imp.to_frame('importance'))

### Leakage check (diagnostic only — not saved)
The labels were built from `facility_type` (mines, industrial) and `lulc_class` (cropland → stubble, forest → wildfire),
and both are also model inputs. If macro F1 collapses without them, the model is mostly reading the labeling rules back,
not learning from the fire's radiometry.

In [ ]:
if RUN_LEAKAGE_CHECK:
    no_leak = [f for f in FEATURE_NAMES if f not in CATEGORICAL]
    p2 = {**params, 'iterations': 1000, 'verbose': 250}
    m2 = CatBoostClassifier(**p2)
    m2.fit(to_pool(train, no_leak), eval_set=to_pool(val, no_leak))
    pred2 = m2.predict(to_pool(val, no_leak)).astype(int).ravel()
    leak_f1 = f1_score(y_val, pred2, average='macro')
    print(f'Macro F1 with land cover + facility type: {macro_f1:.3f}')
    print(f'Macro F1 without them:                   {leak_f1:.3f}')
    print(classification_report(y_val, pred2, labels=labels, target_names=[CLASS_NAMES[c] for c in labels], digits=3))
    del m2
else:
    leak_f1 = None

## 8. Save and verify the CatBoost model

In [ ]:
cb_path = f'{OUT_DIR}/catboost_model.cbm'
model.save_model(cb_path)

chk = CatBoostClassifier()
chk.load_model(cb_path)
assert list(chk.feature_names_) == FEATURE_NAMES, chk.feature_names_
assert [int(c) for c in chk.classes_] == labels
assert tuple(chk.get_cat_feature_indices()) == (9, 10)

# Same probe as the backend: identical radiometry, only land cover changes.
base = [330.0, 295.0, 330 / 295, 12.0, 0.4, 0.4, 1, 0, 50_000.0, 'none', '40.0', 0.5, 0]
for code_ in ['40.0', '10.0', '80.0', '60.0']:
    row = list(base); row[10] = code_
    p = chk.predict_proba(Pool([row], cat_features=[9, 10]))[0]
    print(code_, {CLASS_NAMES[c]: round(float(v), 3) for c, v in zip(labels, p)})
# Inside an industrial polygon: only the site's recent history changes.
for days in [0, 2, 10, 60]:
    row = [330.0, 295.0, 330 / 295, 12.0, 0.4, 0.4, 0, 1, 0.0, 'general_industrial', '50.0', 1.0, days]
    p = chk.predict_proba(Pool([row], cat_features=[9, 10]))[0]
    print(f'industrial site, {days:2d} earlier days burning', {CLASS_NAMES[c]: round(float(v), 3) for c, v in zip(labels, p)})
print('Saved', cb_path, f'({os.path.getsize(cb_path) / 1e6:.1f} MB)')

## 9. Train and save the Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

t = time.time()
iforest = IsolationForest(n_estimators=200, max_samples='auto', contamination=0.01, random_state=SEED, n_jobs=-1)
iforest.fit(if_df[IFOREST_FEATURES])
assert list(iforest.feature_names_in_) == IFOREST_FEATURES, iforest.feature_names_in_
print(f'Fitted in {time.time() - t:.0f}s')

if_path = f'{OUT_DIR}/isolation_forest.joblib'
joblib.dump({'model': iforest, 'features': IFOREST_FEATURES, 'version': 'iforest-1.1.0'}, if_path)

# Backend scaling: anomaly = clip(0.5 - decision_function, 0, 1)
probe = if_df.sample(n=min(100_000, len(if_df)), random_state=SEED)
scores = np.clip(0.5 - iforest.decision_function(probe[IFOREST_FEATURES]), 0, 1)
print('anomaly score percentiles (50/90/99/99.9):', np.round(np.percentile(scores, [50, 90, 99, 99.9]), 3))
print('Saved', if_path, f'({os.path.getsize(if_path) / 1e6:.1f} MB)')

## 10. Training report + zip for download

In [ ]:
import json, datetime

report = {
    'trained_at_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
    'dataset': CSV_NAME,
    'versions': {'catboost': catboost.__version__, 'sklearn': sklearn.__version__, 'joblib': joblib.__version__},
    'gpu': USE_GPU,
    'cap_per_class': cap,
    'train_rows': int(len(train)), 'val_rows': int(len(val)),
    'train_class_counts': {int(k): int(v) for k, v in train['class_id'].value_counts().sort_index().items()},
    'val_class_counts': {int(k): int(v) for k, v in val['class_id'].value_counts().sort_index().items()},
    'catboost': {'feature_names': FEATURE_NAMES, 'best_iteration': int(model.get_best_iteration()),
                 'macro_f1': round(float(macro_f1), 4),
                 'macro_f1_without_landcover_facility': None if leak_f1 is None else round(float(leak_f1), 4),
                 'params': {k: v for k, v in params.items() if k != 'devices'}},
    'isolation_forest': {'features': IFOREST_FEATURES, 'fit_rows': int(len(if_df)), 'version': 'iforest-1.1.0'},
    'classification_report': report_text,
}
with open(f'{OUT_DIR}/training_report.json', 'w') as f:
    json.dump(report, f, indent=2)

zip_path = shutil.make_archive(f'{DRIVE_DIR}/jvalyx_artifacts', 'zip', OUT_DIR)
print('Artifacts:', os.listdir(OUT_DIR))
print('Zip on Drive:', zip_path)

In [ ]:
# Optional: download the zip straight to this computer instead of fetching it from Drive.
from google.colab import files
files.download(zip_path)